In [2]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
words = open('words.txt', 'r').read().splitlines()
words[:8]

['abakus',
 'abandon',
 'abazja',
 'abażur',
 'abażurek',
 'abdukcja',
 'abduktor',
 'abdykacja']

In [4]:
len(words)

48178

In [5]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'r', 18: 's', 19: 't', 20: 'u', 21: 'v', 22: 'w', 23: 'y', 24: 'z', 25: 'ó', 26: 'ą', 27: 'ć', 28: 'ę', 29: 'ł', 30: 'ń', 31: 'ś', 32: 'ź', 33: 'ż', 0: '.'}


In [ ]:
# dataset

block_size = 3  # context size
X, Y = [], []

for w in words[:5]:
    print(w)
    context = [0] * block_size

    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context), '---->', itos[ix])
        context = context[1:] + [ix]  # crop and append

X = torch.tensor(X)
Y = torch.tensor(Y)

abakus
... ----> a
..a ----> b
.ab ----> a
aba ----> k
bak ----> u
aku ----> s
kus ----> .
abandon
... ----> a
..a ----> b
.ab ----> a
aba ----> n
ban ----> d
and ----> o
ndo ----> n
don ----> .
abazja
... ----> a
..a ----> b
.ab ----> a
aba ----> z
baz ----> j
azj ----> a
zja ----> .
abażur
... ----> a
..a ----> b
.ab ----> a
aba ----> ż
baż ----> u
ażu ----> r
żur ----> .
abażurek
... ----> a
..a ----> b
.ab ----> a
aba ----> ż
baż ----> u
ażu ----> r
żur ----> e
ure ----> k
rek ----> .


In [7]:
X.shape

torch.Size([38, 3])

In [8]:
g = torch.Generator().manual_seed(2147483647)

In [9]:
C = torch.randn((34, 2), generator=g)

In [10]:
C

tensor([[ 1.5674, -0.2373],
        [-0.0274, -1.1008],
        [ 0.2859, -0.0296],
        [-1.5471,  0.6049],
        [ 0.0791,  0.9046],
        [-0.4713,  0.7868],
        [-0.3284, -0.4330],
        [ 1.3729,  2.9334],
        [ 1.5618, -1.6261],
        [ 0.6772, -0.8404],
        [ 0.9849, -0.1484],
        [-1.4795,  0.4483],
        [-0.0707,  2.4968],
        [ 2.4448, -0.6701],
        [-1.2199,  0.3031],
        [-1.0725,  0.7276],
        [ 0.0511,  1.3095],
        [-0.8022, -0.8504],
        [-1.8068,  1.2523],
        [-1.2256,  1.2165],
        [-0.9648, -0.2321],
        [-0.3476,  0.3324],
        [-1.3263,  1.1224],
        [ 0.5964,  0.4585],
        [ 0.0540, -1.7400],
        [ 0.1156,  0.8032],
        [-0.8561,  0.5408],
        [ 0.6169,  1.5160],
        [ 0.2472, -0.3777],
        [-1.9081, -0.3717],
        [ 0.1753,  0.9928],
        [-0.6279,  0.0770],
        [-1.9911, -1.3050],
        [-1.3792, -0.3056]])

In [11]:
C[5]

tensor([-0.4713,  0.7868])

In [12]:
F.one_hot(torch.tensor(5), num_classes=34)

tensor([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [13]:
F.one_hot(torch.tensor(5), num_classes=34).float() @ C      # same as ^^ (all the zeros in F.one_hot... cancel values from C except for the 5th row)

tensor([-0.4713,  0.7868])

In [14]:
emb = C[X]
emb.shape

torch.Size([38, 3, 2])

In [15]:
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)

In [16]:
emb.shape, W1.shape     # dimensions don't fit, should be [38,6] and [6, 100]

(torch.Size([38, 3, 2]), torch.Size([6, 100]))

In [17]:
emb = emb.view(-1, 6)   # now okay
emb.shape

torch.Size([38, 6])

In [18]:
h = torch.tanh(emb @ W1 + b1)
h.shape

torch.Size([38, 100])

In [19]:
W2 = torch.randn((100, 34), generator=g)
b2 = torch.randn(34, generator=g)

In [20]:
logits = h @ W2 + b2

In [21]:
logits.shape

torch.Size([38, 34])

In [22]:
counts = logits.exp()

In [23]:
probs = counts / counts.sum(1, keepdim=True)

In [24]:
probs.shape

torch.Size([38, 34])

In [25]:
probs[0].sum()      # sums to 1

tensor(1.0000)

In [26]:
probs[torch.arange(38), Y]

tensor([2.5972e-08, 1.9226e-08, 3.9052e-11, 6.8598e-11, 4.0324e-10, 4.2286e-01,
        6.4735e-13, 2.5972e-08, 1.9226e-08, 3.9052e-11, 7.3672e-07, 1.2764e-10,
        2.7188e-08, 4.1369e-01, 2.7334e-10, 2.5972e-08, 1.9226e-08, 3.9052e-11,
        2.3569e-12, 9.3455e-09, 9.2304e-13, 5.3041e-05, 2.5972e-08, 1.9226e-08,
        3.9052e-11, 1.2172e-07, 6.3278e-08, 6.0942e-05, 2.4163e-09, 2.5972e-08,
        1.9226e-08, 3.9052e-11, 1.2172e-07, 6.3278e-08, 6.0942e-05, 2.7114e-12,
        3.2035e-11, 9.2464e-10])

In [27]:
loss = -probs[torch.arange(38), Y].log().mean()     # inf is i thnk because some of the elements are zero
loss

tensor(18.5749)

In [ ]:
loss = F.cross_entropy(logits, Y)   # numerically stable
loss

tensor(18.5749)

### Summary

In [39]:
# dataset

block_size = 3  # context size
X, Y = [], []

for w in words:
    context = [0] * block_size

    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]  # crop and append

X = torch.tensor(X)
Y = torch.tensor(Y)

In [40]:
X.shape, Y.shape    # dataset

(torch.Size([500441, 3]), torch.Size([500441]))

In [45]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((34, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.rand(100, generator=g)
W2 = torch.randn((100, 34), generator=g)
b2 = torch.rand(34, generator=g)
parameters = [C, W1, b1, W2, b2]

In [46]:
sum(p.nelement() for p in parameters)

4202

In [47]:
for p in parameters:
    p.requires_grad = True

In [ ]:
for _ in range(10):     # long - 4,5s
    #forward pass
    emb = C[X]
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Y)
    print("loss: ", loss)

    #backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    #update
    for p in parameters:
        p.data += -0.1 * p.grad

loss:  tensor(18.7329, grad_fn=<NllLossBackward0>)
loss:  tensor(17.7096, grad_fn=<NllLossBackward0>)
loss:  tensor(16.8167, grad_fn=<NllLossBackward0>)
loss:  tensor(16.1090, grad_fn=<NllLossBackward0>)
loss:  tensor(15.4664, grad_fn=<NllLossBackward0>)
loss:  tensor(14.8687, grad_fn=<NllLossBackward0>)
loss:  tensor(14.3087, grad_fn=<NllLossBackward0>)
loss:  tensor(13.7810, grad_fn=<NllLossBackward0>)
loss:  tensor(13.2829, grad_fn=<NllLossBackward0>)
loss:  tensor(12.8146, grad_fn=<NllLossBackward0>)


In [50]:
torch.randint(0, X.shape[0], (32,))

tensor([214231, 211358, 472857, 278214,   3205, 322822, 143092, 294328, 128191,
        279433, 485633, 391788, 369239,  99859, 453751, 386498, 262554, 471706,
        273489, 173166, 122233, 212487, 164760, 486788,   9000, 220016, 442272,
        462702, 429583, 252050, 372200,  83857])

In [ ]:
for _ in range(10):     # fast - 0s (with minibatches)
    #construct minibatch
    ix = torch.randint(0, X.shape[0], (32,))

    #forward pass
    emb = C[X[ix]]
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Y[ix])
    print("loss: ", loss)

    #backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    #update
    for p in parameters:
        p.data += -0.1 * p.grad

loss:  tensor(11.8478, grad_fn=<NllLossBackward0>)
loss:  tensor(11.6456, grad_fn=<NllLossBackward0>)
loss:  tensor(11.6674, grad_fn=<NllLossBackward0>)
loss:  tensor(11.7256, grad_fn=<NllLossBackward0>)
loss:  tensor(11.1561, grad_fn=<NllLossBackward0>)
loss:  tensor(10.5185, grad_fn=<NllLossBackward0>)
loss:  tensor(9.9563, grad_fn=<NllLossBackward0>)
loss:  tensor(9.6613, grad_fn=<NllLossBackward0>)
loss:  tensor(11.1591, grad_fn=<NllLossBackward0>)
loss:  tensor(9.9788, grad_fn=<NllLossBackward0>)


In [56]:
for _ in range(100):     # fast - 0s (with minibatches)
    #construct minibatch
    ix = torch.randint(0, X.shape[0], (32,))

    #forward pass
    emb = C[X[ix]]
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Y[ix])

    #backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    #update
    for p in parameters:
        p.data += -0.1 * p.grad

print("loss: ", loss.item())

loss:  2.7855896949768066
